# Bab 8: Support Vector Machine

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab08/data_batik.py": "\"\"\"Tekstur sintetis yang terinspirasi empat motif batik (ilustrasi, bukan citra asli).\n\nparang  : garis-garis diagonal miring\nkawung  : susunan lingkaran/elips berulang dalam kisi\ntruntum : titik-titik kecil tersebar acak\nlereng  : pita diagonal lebar bergelombang\nSetiap citra 32 x 32 piksel, dengan variasi skala, rotasi kecil, kontras, dan derau.\n\"\"\"\nimport numpy as np\n\nMOTIF = [\"parang\", \"kawung\", \"truntum\", \"lereng\"]\n\ndef citra(motif, rng, n=32, derau=1.4):\n    y, x = np.mgrid[0:n, 0:n] / n\n    sudut = rng.normal(0, 0.35)\n    xr, yr = x*np.cos(sudut) - y*np.sin(sudut), x*np.sin(sudut) + y*np.cos(sudut)\n    f = rng.uniform(4, 9)\n    if motif == \"parang\":\n        z = np.sin(2*np.pi*f*(xr + yr) / 1.4)\n    elif motif == \"kawung\":\n        u, v = (xr*f/1.6) % 1 - 0.5, (yr*f/1.6) % 1 - 0.5\n        z = np.where(u**2/0.12 + v**2/0.06 < 1, 1.0, -1.0) + np.where(v**2/0.12 + u**2/0.06 < 1, 1.0, 0.0)\n    elif motif == \"truntum\":\n        z = -np.ones((n, n))\n        for cx, cy in rng.uniform(0, n, size=(int(f * 4), 2)).astype(int):\n            z[max(cy-1, 0):cy+2, max(cx-1, 0):cx+2] = 1.0\n    else:\n        z = np.sign(np.sin(2*np.pi*(f/3)*(xr + yr) + 1.5*np.sin(2*np.pi*3*(xr - yr))))\n    z = rng.uniform(0.6, 1.4) * z + rng.normal(0, derau, (n, n))\n    return z\n\ndef fitur(z):\n    \"\"\"Energi spektrum Fourier dalam 4 cincin frekuensi x 4 orientasi = 16 fitur.\"\"\"\n    F = np.abs(np.fft.fftshift(np.fft.fft2(z - z.mean())))**2\n    n = z.shape[0]; ky, kx = np.mgrid[-n//2:n//2, -n//2:n//2]\n    r, th = np.hypot(kx, ky), np.mod(np.arctan2(ky, kx), np.pi)\n    hasil = []\n    for r0, r1 in [(1, 3), (3, 6), (6, 10), (10, 16)]:\n        for k in range(4):\n            m = (r >= r0) & (r < r1) & (th >= k*np.pi/4) & (th < (k+1)*np.pi/4)\n            hasil.append(np.log(F[m].sum() + 1e-9))\n    return np.array(hasil)\n\ndef buat_data(n_per_motif=150, seed=0):\n    rng = np.random.default_rng(seed)\n    X, y = [], []\n    for k, m in enumerate(MOTIF):\n        for _ in range(n_per_motif):\n            X.append(fitur(citra(m, rng))); y.append(k)\n    return np.array(X), np.array(y)\n",
"bab08/svm_batik.py": "import numpy as np\nfrom sklearn.svm import SVC\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import GridSearchCV, train_test_split\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import confusion_matrix\nfrom data_batik import buat_data, MOTIF\n\nX, y = buat_data()\nXl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)\n\npipa = make_pipeline(StandardScaler(), SVC(kernel=\"rbf\"))                 #<1>\nkisi = {\"svc__C\": [0.1, 1, 10, 100], \"svc__gamma\": [0.01, 0.03, 0.1, 0.3]}\ncari = GridSearchCV(pipa, kisi, cv=5).fit(Xl, yl)                        #<2>\nprint(f\"terbaik: {cari.best_params_} | akurasi CV {cari.best_score_:.3f}\")\nsvm = cari.best_estimator_\nprint(f\"SVM RBF         : akurasi uji {svm.score(Xu, yu):.3f} | \"\n      f\"support vector {svm[-1].n_support_.sum()} dari {len(yl)}\")    #<3>\nfor nama, m in [(\"SVM linear\", SVC(kernel=\"linear\")),\n                (\"regresi logistik\", LogisticRegression(max_iter=3000))]:\n    m = make_pipeline(StandardScaler(), m).fit(Xl, yl)\n    print(f\"{nama:16s}: akurasi uji {m.score(Xu, yu):.3f}\")\nprint(\"kebingungan SVM RBF, baris = sebenarnya:\", \", \".join(MOTIF))\nprint(confusion_matrix(yu, svm.predict(Xu)))\n",
"bab08/svm_margin.py": "import numpy as np\nfrom sklearn.svm import SVC\nfrom sklearn.preprocessing import StandardScaler\nfrom data_batik import buat_data\n\nX, y = buat_data()\npilih = (y == 0) | (y == 1)                         # parang vs kawung  #<1>\nX2 = StandardScaler().fit_transform(X[pilih][:, [6, 7]])   # dua fitur spektrum\nt = np.where(y[pilih] == 1, 1, -1)                  # label SVM: -1 dan +1\n\nprint(\"      C | vektor pend. | margin | slack | akurasi latih\")\nfor C in (1000, 1, 0.01):                           # dari margin keras ke lunak  #<2>\n    svm = SVC(kernel=\"linear\", C=C).fit(X2, t)\n    w = svm.coef_[0]\n    lebar = 2 / np.linalg.norm(w)                   # lebar margin  #<3>\n    xi = np.maximum(0, 1 - t * svm.decision_function(X2))   # variabel slack\n    print(f\"{C:7.2f} | {len(svm.support_):3d} dari {len(t)} | {lebar:6.2f} | \"\n          f\"{xi.sum():5.1f} | {svm.score(X2, t):.3f}\")\n",
"bab08/svr_rvm.py": "import numpy as np\nfrom sklearn.svm import SVR\n\nrng = np.random.default_rng(0)\nx = np.sort(rng.uniform(-10, 10, 100)); t = np.sinc(x / np.pi) + rng.normal(0, 0.1, 100)\nxu = np.linspace(-10, 10, 500); fu = np.sinc(xu / np.pi)       # sin(x)/x\nK = lambda a, b: np.exp(-(a[:, None] - b[None, :])**2 / (2 * 1.5**2))\n\nsvr = SVR(kernel=\"rbf\", gamma=1 / (2 * 1.5**2), C=10, epsilon=0.1).fit(x[:, None], t)  #<1>\nr_svr = np.sqrt(np.mean((svr.predict(xu[:, None]) - fu)**2))\n\n# Relevance vector machine: satu presisi prior alpha_i per basis  #<2>\nPhi = np.c_[np.ones(100), K(x, x)]\nalpha, beta = np.ones(Phi.shape[1]), 100.0\nfor _ in range(300):\n    aktif = alpha < 1e6                                        # basis yang belum dipangkas\n    P = Phi[:, aktif]\n    S = np.linalg.inv(np.diag(alpha[aktif]) + beta * P.T @ P)\n    m = beta * S @ P.T @ t\n    gamma = 1 - alpha[aktif] * np.diag(S)\n    alpha[aktif] = gamma / m**2                                # re-estimasi seperti Bab 4  #<3>\n    beta = (100 - gamma.sum()) / np.sum((t - P @ m)**2)\naktif = alpha < 1e6\nr_rvm = np.sqrt(np.mean((np.c_[np.ones(500), K(xu, x)][:, aktif] @ m - fu)**2))\nprint(f\"SVR: {len(svr.support_):3d} support vector   | RMSE {r_svr:.3f}\")\nprint(f\"RVM: {aktif.sum() - aktif[0]:3d} vektor relevansi  | RMSE {r_rvm:.3f} | \"\n      f\"sd derau {1/np.sqrt(beta):.3f} (sebenarnya 0.100)\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab08')
print('folder kerja:', os.getcwd())

## Listing 8.1: Margin keras dan lunak untuk parang lawan kawung

Berkas: `kode/bab08/svm_margin.py`

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from data_batik import buat_data

X, y = buat_data()
pilih = (y == 0) | (y == 1)                         # parang vs kawung  # (1)
X2 = StandardScaler().fit_transform(X[pilih][:, [6, 7]])   # dua fitur spektrum
t = np.where(y[pilih] == 1, 1, -1)                  # label SVM: -1 dan +1

print("      C | vektor pend. | margin | slack | akurasi latih")
for C in (1000, 1, 0.01):                           # dari margin keras ke lunak  # (2)
    svm = SVC(kernel="linear", C=C).fit(X2, t)
    w = svm.coef_[0]
    lebar = 2 / np.linalg.norm(w)                   # lebar margin  # (3)
    xi = np.maximum(0, 1 - t * svm.decision_function(X2))   # variabel slack
    print(f"{C:7.2f} | {len(svm.support_):3d} dari {len(t)} | {lebar:6.2f} | "
          f"{xi.sum():5.1f} | {svm.score(X2, t):.3f}")

## Listing 8.2: SVM RBF untuk empat motif, dengan pencarian kisi

Berkas: `kode/bab08/svm_batik.py`

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from data_batik import buat_data, MOTIF

X, y = buat_data()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

pipa = make_pipeline(StandardScaler(), SVC(kernel="rbf"))                 # (1)
kisi = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.01, 0.03, 0.1, 0.3]}
cari = GridSearchCV(pipa, kisi, cv=5).fit(Xl, yl)                        # (2)
print(f"terbaik: {cari.best_params_} | akurasi CV {cari.best_score_:.3f}")
svm = cari.best_estimator_
print(f"SVM RBF         : akurasi uji {svm.score(Xu, yu):.3f} | "
      f"support vector {svm[-1].n_support_.sum()} dari {len(yl)}")    # (3)
for nama, m in [("SVM linear", SVC(kernel="linear")),
                ("regresi logistik", LogisticRegression(max_iter=3000))]:
    m = make_pipeline(StandardScaler(), m).fit(Xl, yl)
    print(f"{nama:16s}: akurasi uji {m.score(Xu, yu):.3f}")
print("kebingungan SVM RBF, baris = sebenarnya:", ", ".join(MOTIF))
print(confusion_matrix(yu, svm.predict(Xu)))

## Listing 8.3: SVR dan relevance vector machine pada data sinc

Berkas: `kode/bab08/svr_rvm.py`

In [ ]:
import numpy as np
from sklearn.svm import SVR

rng = np.random.default_rng(0)
x = np.sort(rng.uniform(-10, 10, 100)); t = np.sinc(x / np.pi) + rng.normal(0, 0.1, 100)
xu = np.linspace(-10, 10, 500); fu = np.sinc(xu / np.pi)       # sin(x)/x
K = lambda a, b: np.exp(-(a[:, None] - b[None, :])**2 / (2 * 1.5**2))

svr = SVR(kernel="rbf", gamma=1 / (2 * 1.5**2), C=10, epsilon=0.1).fit(x[:, None], t)  # (1)
r_svr = np.sqrt(np.mean((svr.predict(xu[:, None]) - fu)**2))

# Relevance vector machine: satu presisi prior alpha_i per basis  # (2)
Phi = np.c_[np.ones(100), K(x, x)]
alpha, beta = np.ones(Phi.shape[1]), 100.0
for _ in range(300):
    aktif = alpha < 1e6                                        # basis yang belum dipangkas
    P = Phi[:, aktif]
    S = np.linalg.inv(np.diag(alpha[aktif]) + beta * P.T @ P)
    m = beta * S @ P.T @ t
    gamma = 1 - alpha[aktif] * np.diag(S)
    alpha[aktif] = gamma / m**2                                # re-estimasi seperti Bab 4  # (3)
    beta = (100 - gamma.sum()) / np.sum((t - P @ m)**2)
aktif = alpha < 1e6
r_rvm = np.sqrt(np.mean((np.c_[np.ones(500), K(xu, x)][:, aktif] @ m - fu)**2))
print(f"SVR: {len(svr.support_):3d} support vector   | RMSE {r_svr:.3f}")
print(f"RVM: {aktif.sum() - aktif[0]:3d} vektor relevansi  | RMSE {r_rvm:.3f} | "
      f"sd derau {1/np.sqrt(beta):.3f} (sebenarnya 0.100)")